In [36]:
# importando os scripts de consulta

import sys
import os

# Altera o caminho para a pasta raiz do projeto (um nível acima)
raiz_projeto = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Adiciona a raiz do projeto ao caminho de busca do Python se ainda não estiver lá
if raiz_projeto not in sys.path:
    sys.path.append(raiz_projeto)

# Agora você pode importar os módulos da pasta scripts usando a notação de ponto
from scripts.crm_leads import load_leads




In [37]:
leads = load_leads()


Página 1: 250 registros
Página 2: 250 registros
Página 3: 250 registros
Página 4: 250 registros
Página 5: 250 registros
Página 6: 250 registros
Página 7: 250 registros
Página 8: 250 registros
Página 9: 250 registros
Página 10: 250 registros
Página 11: 250 registros
Página 12: 250 registros
Página 13: 250 registros
Página 14: 250 registros
Página 15: 250 registros
Página 16: 250 registros
Página 17: 250 registros
Página 18: 250 registros
Página 19: 250 registros
Página 20: 250 registros
Página 21: 250 registros
Página 22: 250 registros
Página 23: 250 registros
Página 24: 250 registros
Página 25: 250 registros
Página 26: 4 registros


In [38]:
"""
CRM Educacional -> Power BI
Script refatorado (template)

Preencha a lista `atributos` e o dicionário `rename_columns`
com os valores do seu projeto.
"""

import time
import requests
import xmltodict
import pandas as pd

BASE_URL = "https://trivento.api.crmeducacional.com"
URL = f"{BASE_URL}/api/LocalizacaoAvancada"

USERNAME = "IntegraçãoCRM"
PASSWORD = "IntegraçãoCRM"
AUTH_TYPE = "APICRMEducacional"

atributos = [
    'cad_numeroinscricao',
    "cad_clientepotencialid",
    "crmeduc_data_aprovacao",
    "cad_datapagamento",
    "cad_situacaoavaliacao",
    "cad_situacao",
    "cad_ofertaid",
    "trivento_unidadedeensinotrivento",
    "cad_concursoid",
    "coc_situacaopagamento",
]

rename_columns = {
    'cad_inscricaocandidatoid': 'id_inscricao',
    "cad_clientepotencialid": "ID Lead",
    'cad_numeroinscricao': 'Numero Inscricao',
    "crmeduc_data_aprovacao": "Data Aprovação",
    "cad_datapagamento": "Data de Pagamento",
    "cad_situacaoavaliacao": "Situação de Avaliação",
    "cad_situacao": "Situação Inscrição",
    "cad_ofertaid": "Curso ID",
    "trivento_unidadedeensinotrivento": "Unidade Inscrito",
    "cad_concursoid": "Forma de Ingresso",
    "coc_situacaopagamento": "Situação Pagamento",
}


def get_token(session):
    payload = {
        "grant_type": "password",
        "username": USERNAME,
        "password": PASSWORD,
        "authentication_type": AUTH_TYPE,
    }

    response = session.post(
        f"{BASE_URL}/api/token",
        data=payload,
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        timeout=30,
    )
    response.raise_for_status()
    return response.json()["access_token"]


def create_headers(token):
    return {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/xml",
    }


def create_fetch_xml(attributes):
    tags = "\n".join(
        f'<attribute name="{c}"/>' for c in attributes
    )

    return f"""
<fetch version="1.0" output-format="xml-platform" mapping="logical">
  <entity name="cad_inscricaocandidato">
    {tags}
    <order attribute="createdon" descending="true"/>
  </entity>
</fetch>
"""


def post_retry(session, *args, retries=3, **kwargs):
    for i in range(retries):
        try:
            r = session.post(*args, **kwargs)
            r.raise_for_status()
            return r
        except requests.RequestException:
            if i == retries - 1:
                raise
            time.sleep(5)


def get_page(session, page, headers, body):
    headers = headers.copy()
    headers["Pagina"] = str(page)

    response = post_retry(
        session,
        URL,
        headers=headers,
        data=body.encode("utf-8"),
        timeout=60,
    )

    doc = xmltodict.parse(response.text)

    resultset = doc["resultset"]
    records = resultset["result"]

    if isinstance(records, dict):
        records = [records]

    return records, resultset["@morerecords"] == "1"


def get_all_records(session, headers, body):
    page = 1
    rows = []

    while True:
        records, more = get_page(session, page, headers, body)
        rows.extend(records)
        print(f"Página {page}: {len(records)} registros")

        if not more:
            break

        page += 1

    return rows


def flatten_value(value):
    if not isinstance(value, dict):
        return value

    if "@name" in value:
        return value["@name"]

    if "#text" in value:
        return value["#text"]

    return value


def flatten_record(record):
    return {
        k: v['#text'] if k == 'cad_clientepotencialid' else  flatten_value(v)
        for k, v in record.items()
    }


def load_inscricoes():

    with requests.Session() as session:

        token = get_token(session)
        headers = create_headers(token)
        body = create_fetch_xml(atributos)

        records = get_all_records(
            session,
            headers,
            body,
        )
    
    df = pd.DataFrame(
        flatten_record(r)
        for r in records
    )

    for col in ["'createdon'", "modifiedon"]:
        if col in df.columns:
            df[col] = pd.to_datetime(
                df[col],
                format="%d/%m/%Y %H:%M",
                errors="coerce",
            )

    df.rename(
        columns={
            k: v
            for k, v in rename_columns.items()
            if k in df.columns
        },
        inplace=True,
    )

    map = {
        "Faculdade Serra Dourada de " : "",
        "Faculdade Alis" : "Itabirito",
        " dos Carajás" : ""}
    
    df['Unidade Inscrito'] = df['Unidade Inscrito'].map(map)
    return df

inscricoes = load_inscricoes()





Página 1: 250 registros
Página 2: 250 registros
Página 3: 250 registros
Página 4: 250 registros
Página 5: 250 registros
Página 6: 250 registros
Página 7: 250 registros
Página 8: 250 registros
Página 9: 250 registros
Página 10: 250 registros
Página 11: 250 registros
Página 12: 250 registros
Página 13: 222 registros


In [39]:
inscricoes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3222 entries, 0 to 3221
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Numero Inscricao       3222 non-null   object
 1   ID Lead                3222 non-null   object
 2   Situação de Avaliação  3222 non-null   object
 3   Situação Inscrição     3222 non-null   object
 4   Curso ID               3222 non-null   object
 5   Unidade Inscrito       514 non-null    object
 6   Forma de Ingresso      3222 non-null   object
 7   createdon              3222 non-null   object
 8   id_inscricao           3222 non-null   object
 9   Data Aprovação         1097 non-null   object
 10  Situação Pagamento     1287 non-null   object
 11  Data de Pagamento      664 non-null    object
dtypes: object(12)
memory usage: 302.2+ KB


In [40]:
print(f'Total de leads: {leads.shape[0]}')
print(f'Total de inscricoes: {inscricoes.shape[0]}')

Total de leads: 6254
Total de inscricoes: 3222


In [41]:
leads = leads[['ID Lead', 'Nome Completo']]

In [42]:
inscricoes = inscricoes[['ID Lead', 'id_inscricao']]

In [43]:
df = leads.merge(inscricoes, how= 'left', on = 'ID Lead')

In [44]:
df.head()

,ID Lead,Nome Completo,id_inscricao
0,{C00217C4-7A88-F111-8077-6045BD39BE00},Ivani Esteves de Melo,NaN
1,{CB180AA1-7A88-F111-8077-6045BD39BE00},Juran Sousa de Sousa,NaN
2,{110BFE81-7988-F111-8077-6045BD39BE00},Carlos Daniel,NaN
3,{C24CD7A6-7588-F111-8077-6045BD39BC63},Elder Rocha Madeira,{4D8E86C2-7588-F111-8077-6045BD39BC63}
4,{487A214C-7488-F111-8077-000D3AC0F764},Charles Vinicius da Silva Costa,{F755997C-7488-F111-8077-000D3AC0F764}


In [45]:
colunas = ['ID Lead','id_inscricao']

for coluna in colunas:
    df[coluna] = df[coluna].str.strip().str.lower()
    df[coluna] = df[coluna].str.replace('{', '').str.replace('}', '')

In [46]:
df['Nome Completo'] = df['Nome Completo'].str.strip().str.lower()

In [47]:
testes = df[df['Nome Completo'].str.contains('teste')]

In [48]:
testes.shape

(107, 3)

In [50]:
testes.sample(10)

,ID Lead,Nome Completo,id_inscricao
1706,69e6a318-007a-f111-ab0e-000d3ac0f764,teste crm educacional,ebc11f1f-007a-f111-ab0e-000d3ac0f764
1030,8522578b-c47e-f111-ab0e-6045bd39bc63,adauri teste,NaN
1371,f0b17e98-d47b-f111-ab0e-7ced8db11fb7,bridha teste,0cb27e98-d47b-f111-ab0e-7ced8db11fb7
2980,2034d175-b874-f111-ab0e-00224836b90d,karolaine teste,b4c6777c-b874-f111-ab0e-00224836b90d
191,e0e0d82d-c186-f111-ab0f-00224836b90d,barbara teste,adaff733-c186-f111-ab0f-00224836b90d
1778,1aae9a90-7d79-f111-ab0e-00224836b90d,arthur teste,97c4d00a-7e79-f111-ab0e-00224836b90d
3478,0553e0d7-d670-f111-ab0f-000d3ac0f764,teste paggg,13bb3cdf-d670-f111-ab0f-000d3ac0f764
134,abd88701-5887-f111-ab0f-00224836b90d,bárbara teste,29469807-5887-f111-ab0f-00224836b90d
1577,2987898c-2c7a-f111-ab0e-6045bd39bc63,carol teste,NaN
2352,df5ae0cd-ee76-f111-ab0e-7ced8db11fb7,teste cindo,4a72d1d9-ee76-f111-ab0e-7ced8db11fb7


In [51]:
testes.to_csv('dataset.csv', index= False)

In [53]:
testes.loc[testes['ID Lead'] == '69e6a318-007a-f111-ab0e-000d3ac0f764', 'id_inscricao'].values

array(['ebc11f1f-007a-f111-ab0e-000d3ac0f764'], dtype=object)